# Nightly Report: IQ, AOS, Thermal, Aberrations and Degrees of Freedom

Owner: **Guillem Megias** <br>
Last Verified to Run: **2025-07-07** <br>

In [ ]:
# Times Square Parameters
day_obs = 20250706
seq_min = 100
seq_max = 200

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd
from matplotlib.dates import DateFormatter

from lsst.ts.aos.analysis import AOSDatabase

%matplotlib inline

In [ ]:

zk_groups = [[0], [11 - 4], [1, 2], [3,4], [5, 6]]
zk_group_labels = ['Z4', 'Z11', 'Z5 / Z6', 'Z7 / Z8', ' Z9 / Z10']

groups = [[0], [5], [1, 2, 6, 7], [3, 4], [8,9]]
group_labels = [' M2 dz', 'Cam dz', 'Decenters', 'M2 tilts', 'Cam tilts']
labels = ['m2 dz', 'm2 dx', 'm2 dy', 'm2 rx', 'm2 ry',
          'cam dz', 'cam dx', 'cam dy', 'cam rx', 'cam ry']


mirror_groups = [[10, 11, 30, 31], [12, 34], [13, 14, 32, 33], [15, 16, 35, 36], [17, 18, 37, 38]]
mirror_group_labels = ['Astig', 'Spherical', 'Trefoil', 'Coma', 'Quad']
all_labels = ['M2 dz', 'M2 dx', 'M2 dy', 'M2 rx', 'M2 ry',
     'cam dz', 'cam dx', 'cam dy', 'cam rx', 'cam ry',
     '$B_{{1,1}}$', '$B_{{1,2}}$', '$B_{{1,3}}$', '$B_{{1,4}}$', '$B_{{1,5}}$',
     '$B_{{1,6}}$', '$B_{{1,7}}$', '$B_{{1,8}}$', '$B_{{1,9}}$', '$B_{{1,10}}$',
     '$B_{{1,11}}$', '$B_{{1,12}}$', '$B_{{1,13}}$', '$B_{{1,14}}$', '$B_{{1,15}}$',
     '$B_{{1,16}}$', '$B_{{1,17}}$', '$B_{{1,18}}$', '$B_{{1,19}}$', '$B_{{1,20}}$',
     '$B_{{2,1}}$', '$B_{{2,2}}$', '$B_{{2,3}}$', '$B_{{2,4}}$', '$B_{{2,5}}$',
     '$B_{{2,6}}$', '$B_{{2,7}}$', '$B_{{2,8}}$', '$B_{{2,9}}$', '$B_{{2,10}}$',
     '$B_{{2,11}}$', '$B_{{2,12}}$', '$B_{{2,13}}$', '$B_{{2,14}}$', '$B_{{2,15}}$',
     '$B_{{2,16}}$', '$B_{{2,17}}$', '$B_{{2,18}}$', '$B_{{2,19}}$', '$B_{{2,20}}$'
     ]

## Simplified Nightly Report

In [ ]:
db = AOSDatabase(day_obs=day_obs, seq_min=seq_min, seq_max=seq_max)
await db.create(simplified=True)
table = db.table

In [ ]:
filtered_table = table[table['day_obs'] == day_obs]
raw_filtered_table = table[table['day_obs'] == day_obs]
filtered_table = filtered_table.select_dtypes(include="number")
filtered_table = filtered_table.groupby("seq").mean().reset_index()

# -- AOS jumps
aos_diff = filtered_table["aos_fwhm"].diff()
jump_indices = filtered_table["seq"][aos_diff > 0.3].tolist()

# -- States array and labels
states_per_seq = (
    raw_filtered_table[["seq", "zernikes_fwhm"]]
    .drop_duplicates("seq")
    .dropna(subset=["zernikes_fwhm"])
    .set_index("seq")
)


zernikes_fwhm = np.vstack(states_per_seq["zernikes_fwhm"].values)
seqs = states_per_seq.index.values

In [ ]:
fig = plt.figure(figsize=(30, 15))


linewidth = 0.7
# Top 5x4 GridSpec occupies the upper half
gs_top = gridspec.GridSpec(
    nrows=5, ncols=4,
    width_ratios=[4, 2, 2, 2],
    height_ratios=[1]*5,
    hspace=0.0,
    wspace=0.25,
    top=0.97,
    bottom=0.54  # ends halfway down
)

# Bottom 5x4 GridSpec occupies the lower half
gs_bot = gridspec.GridSpec(
    nrows=5, ncols=4,
    width_ratios=[4, 2, 2, 2],
    height_ratios=[1]*5,
    hspace=0.0,
    top=0.46,
    bottom=0.05
)

# -- Left column: Survey performance
axes = []
for i in range(5):
    ax = fig.add_subplot(gs_top[i, 0], sharex=axes[0] if i > 0 else None)
    axes.append(ax)
axes[0].scatter(filtered_table['seq'], filtered_table['fwhm_zenith_500nm'], s=3, label='FWHM')
axes[0].scatter(filtered_table['seq'], filtered_table['dimm'], s=3, label='DIMM')
axes[0].legend()
axes[0].set_ylabel('FWHM [arcsec]')
axes[0].set_title(f'Delivered Seeing and System Variables')
axes[1].scatter(filtered_table['seq'], filtered_table["aos_fwhm"], s=3)
axes[2].scatter(filtered_table['seq'], filtered_table['elevation'], color='k', s=3)
axes[3].scatter(filtered_table['seq'], filtered_table['azimuth'], color='k', s=3)

axes[0].set_ylabel('FWHM\n[arcsec]')
axes[1].set_ylabel('AOS FWHM\n[arcsec]')
axes[2].set_ylabel('Elevation\n[deg]')
axes[3].set_ylabel('Azimuth\n[deg]')
axes[3].set_xlabel('Sequence Number')


for ax in axes[1::]:
    for x in jump_indices:
        ax.axvline(x=x, color='red', linestyle='--', linewidth=linewidth)
for ax in axes[:-1]:
    ax.tick_params(labelbottom=False)
for ax in axes:
    ax.grid(True, alpha=0.5)
    ax.tick_params(direction='in', which='both')


# -- Right column: DoF plots with shared x-axis (not y)
axes = [fig.add_subplot(gs_top[i, 1]) for i in range(5)]
for id_group, (ax, zk_group) in enumerate(zip(axes, zk_groups)):
    for zk_idx, i in enumerate(zk_group):
        color = 'black' if zk_idx == 0 else 'gray' if zk_idx == 1 else None
        ax.scatter(seqs, zernikes_fwhm[:, i], s=5, color=color)
    ax.set_ylabel(zk_group_labels[id_group])
    ax.grid(True, alpha=0.5)
    ax.tick_params(direction="in")

for ax in axes[:-1]:
    ax.tick_params(labelbottom=False)
    ax.grid(True, alpha=0.5)
for ax in axes:
    for x in jump_indices:
        ax.axvline(x=x, color='red', linestyle='--', linewidth=linewidth)
axes[-1].set_xlabel("Sequence Number")
axes[0].set_title(f'Optical Aberrations')


plt.tight_layout(rect=[0, 0, 1, 1])
fig.suptitle("Survey Mode Performance Simplified – Day " + str(day_obs), fontsize=18, y=1.03)

## Full Nightly Report

In [ ]:
db = AOSDatabase(day_obs=day_obs, seq_min=seq_min, seq_max=seq_max)
await db.create(simplified=True)
table = db.table

In [ ]:
db.seq_max = 900
await db.update()
table = db.table

In [ ]:
filtered_table = table[table['day_obs'] == day_obs]
raw_filtered_table = table[table['day_obs'] == day_obs]
filtered_table = filtered_table.select_dtypes(include="number")
filtered_table = filtered_table.groupby("seq").mean().reset_index()

# -- AOS jumps
aos_diff = filtered_table["aos_fwhm"].diff()
jump_indices = filtered_table["seq"][aos_diff > 0.3].tolist()

# -- States array and labels
states_per_seq = (
    raw_filtered_table[["seq", "dof_state", "residual_dof_state", "zernikes_fwhm", "lut_state"]]
    .drop_duplicates("seq")
    .dropna(subset=["dof_state", "residual_dof_state", "zernikes_fwhm", "lut_state"])
    .set_index("seq")
)

dof_state = np.vstack(states_per_seq["dof_state"].values)
residual_dof_state = np.vstack(states_per_seq["residual_dof_state"].values)
zernikes_fwhm = np.vstack(states_per_seq["zernikes_fwhm"].values)
lut_state = np.vstack(states_per_seq["lut_state"].values)
seqs = states_per_seq.index.values

In [ ]:
fig = plt.figure(figsize=(30, 15))


linewidth = 0.7
# Top 5x4 GridSpec occupies the upper half
gs_top = gridspec.GridSpec(
    nrows=5, ncols=4,
    width_ratios=[4, 2, 2, 2],
    height_ratios=[1]*5,
    hspace=0.0,
    wspace=0.25,
    top=0.97,
    bottom=0.54  # ends halfway down
)

# Bottom 5x4 GridSpec occupies the lower half
gs_bot = gridspec.GridSpec(
    nrows=5, ncols=4,
    width_ratios=[4, 2, 2, 2],
    height_ratios=[1]*5,
    hspace=0.0,
    top=0.46,
    bottom=0.05
)

# -- Left column: Survey performance
axes = []
for i in range(5):
    ax = fig.add_subplot(gs_top[i, 0], sharex=axes[0] if i > 0 else None)
    axes.append(ax)
axes[0].scatter(filtered_table['seq'], filtered_table['fwhm_zenith_500nm'], s=3, label='FWHM')
axes[0].scatter(filtered_table['seq'], filtered_table['ringss'], s=3, label='RINGSS')
axes[0].legend()
axes[0].set_ylabel('FWHM [arcsec]')
axes[0].set_title(f'Delivered Seeing and System Variables')
axes[1].scatter(filtered_table['seq'], filtered_table["aos_fwhm"], s=3)
axes[2].scatter(filtered_table['seq'], filtered_table['elevation'], color='k', s=3)
axes[3].scatter(filtered_table['seq'], filtered_table['azimuth'], color='k', s=3)
axes[4].scatter(filtered_table['seq'], filtered_table['rotation_angle'], color='k', s=3)

axes[0].set_ylabel('FWHM\n[arcsec]')
axes[1].set_ylabel('AOS FWHM\n[arcsec]')
axes[2].set_ylabel('Elevation\n[deg]')
axes[3].set_ylabel('Azimuth\n[deg]')
axes[4].set_ylabel('Rotator\n[deg]')
axes[4].set_xlabel('Sequence Number')


for ax in axes[1::]:
    for x in jump_indices:
        ax.axvline(x=x, color='red', linestyle='--', linewidth=linewidth)
for ax in axes[:-1]:
    ax.tick_params(labelbottom=False)
for ax in axes:
    ax.grid(True, alpha=0.5)
    ax.tick_params(direction='in', which='both')

# Thermal gradients 
axes = [fig.add_subplot(gs_bot[i, 0]) for i in range(4)]
axes[0].scatter(filtered_table['seq'], filtered_table['m1m3_delta_t'], color='k', s=3)
axes[1].scatter(filtered_table['seq'], filtered_table['m2_delta_t'], color='k', s=3)
axes[2].scatter(filtered_table['seq'], filtered_table['cam_hex_m1m3_delta_t'], color='k', s=3)
axes[3].scatter(filtered_table['seq'], filtered_table['dome_delta_t'], color='k', s=3)

axes[0].set_ylabel(f'M1M3 $\Delta T$\n[C]')
axes[1].set_ylabel('M2 - amb\n[C]')
axes[2].set_ylabel('Cam - M1M3\n[C]')
axes[3].set_ylabel('In - Out\n[C]')
axes[3].set_xlabel('Sequence Number')

for ax in axes:
    for x in jump_indices:
        ax.axvline(x=x, color='red', linestyle='--', linewidth=linewidth)
for ax in axes[:-1]:
    ax.tick_params(labelbottom=False)
for ax in axes:
    ax.grid(True, alpha=0.5)
    ax.tick_params(direction='in', which='both')
axes[0].set_title(f'Thermal Gradients')

# -- Right column: DoF plots with shared x-axis (not y)
axes = [fig.add_subplot(gs_top[i, 1]) for i in range(5)]
for id_group, (ax, zk_group) in enumerate(zip(axes, zk_groups)):
    for zk_idx, i in enumerate(zk_group):
        color = 'black' if zk_idx == 0 else 'gray' if zk_idx == 1 else None
        ax.scatter(seqs, zernikes_fwhm[:, i], s=5, color=color)
    ax.set_ylabel(zk_group_labels[id_group])
    ax.grid(True, alpha=0.5)
    ax.tick_params(direction="in")

for ax in axes[:-1]:
    ax.tick_params(labelbottom=False)
    ax.grid(True, alpha=0.5)
for ax in axes:
    for x in jump_indices:
        ax.axvline(x=x, color='red', linestyle='--', linewidth=linewidth)
axes[-1].set_xlabel("Sequence Number")
axes[0].set_title(f'Optical Aberrations')


# -- Right column: DoF plots with shared x-axis (not y)
axes = [fig.add_subplot(gs_bot[i, 2]) for i in range(5)]
for id_group, (ax, dof_group) in enumerate(zip(axes, groups)):
    for i in dof_group:
        ax.scatter(seqs, lut_state[:, i] + dof_state[:, i], label=f"{labels[i]}", s=3)
    ax.set_ylabel(group_labels[id_group])
    ax.grid(True, alpha=0.5)
    ax.tick_params(direction="in")

for ax in axes[:-1]:
    ax.tick_params(labelbottom=False)
    ax.grid(True, alpha=0.5)
for ax in axes:
    for x in jump_indices:
        ax.axvline(x=x, color='red', linestyle='--', linewidth=linewidth)
axes[0].set_title(f'Hexapods State (LUT + trim)')
axes[-1].set_xlabel("Sequence Number")



# -- Right column: DoF plots with shared x-axis (not y)
axes = [fig.add_subplot(gs_bot[i, 3]) for i in range(5)]
for id_group, (ax, dof_group) in enumerate(zip(axes, mirror_groups)):
    for i in dof_group:
        #ax.scatter(seqs, dof_state[:, i], label=f"{all_labels[i]}", s=3)        
        ax.scatter(seqs, dof_state[:, i]+lut_state[:, i], label=f"{all_labels[i]}", s=3)
    ax.set_ylabel(mirror_group_labels[id_group])
    ax.grid(True, alpha=0.5)
    ax.tick_params(direction="in")
    ax.legend(
        loc='center left',
        bbox_to_anchor=(1.01, 0.5),
        borderaxespad=-0.75,
        frameon=False,
        fontsize=10,
        handletextpad=0.1
    )

for ax in axes[:-1]:
    ax.tick_params(labelbottom=False)
    ax.grid(True, alpha=0.5)
for ax in axes:
    for x in jump_indices:
        ax.axvline(x=x, color='red', linestyle='--', linewidth=linewidth)
axes[-1].set_xlabel("Sequence Number")
axes[0].set_title(f'Mirror State (LUT + trim)')


# -- Right column: DoF plots with shared x-axis (not y)
axes = [fig.add_subplot(gs_top[i, 2]) for i in range(5)]
for id_group, (ax, dof_group) in enumerate(zip(axes, groups)):
    for i in dof_group:
        ax.scatter(seqs, residual_dof_state[:, i], label=f"{labels[i]}", s=3)
    ax.set_ylabel(group_labels[id_group])
    ax.grid(True, alpha=0.5)
    ax.tick_params(direction="in")

for ax in axes[:-1]:
    ax.tick_params(labelbottom=False)
    ax.grid(True, alpha=0.5)
for ax in axes:
    for x in jump_indices:
        ax.axvline(x=x, color='red', linestyle='--', linewidth=linewidth)
axes[0].set_title(f'Residual Hexapod State')
axes[-1].set_xlabel("Sequence Number")


axes = [fig.add_subplot(gs_top[i, 3]) for i in range(5)]
for id_group, (ax, dof_group) in enumerate(zip(axes, mirror_groups)):
    for i in dof_group:
        ax.scatter(seqs, residual_dof_state[:, i], label=f"{all_labels[i]}", s=3)
    ax.set_ylabel(mirror_group_labels[id_group])
    ax.grid(True, alpha=0.5)
    ax.tick_params(direction="in")
    ax.legend(
        loc='center left',
        bbox_to_anchor=(1.01, 0.5),
        borderaxespad=-0.75,
        frameon=False,
        fontsize=10,
        handletextpad=0.1
    )

for ax in axes[:-1]:
    ax.tick_params(labelbottom=False)
    ax.grid(True, alpha=0.5)
for ax in axes:
    for x in jump_indices:
        ax.axvline(x=x, color='red', linestyle='--', linewidth=linewidth)
axes[-1].set_xlabel("Sequence Number")
axes[0].set_title(f'Residual Mirror State')




plt.tight_layout(rect=[0, 0, 1, 1])
fig.suptitle("Survey Mode Performance and AOS DoFs – Day " + str(day_obs), fontsize=18, y=1.03)